# Week 2b.2 — Agent Basics in LangChain

Now that we've covered the API basics in LangChain, we're ready to implement our first agent.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

API key loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")


## 1. The problem

LLMs can serve as the brain but their knowledge and reach are limited.

In [4]:
from pprint import pprint

response = model.invoke("What time is it right now?")
pprint(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'extras': {'signature': 'El4KXAFpFH0TfVuTuAGOcotigacNlz2ibn13rmC/q7CLm9LPUgla5Ci8Y/rhlt9lFtKlx4bvcCEpVoZtAJAxCk7qWz04qIH6Rh+tFnASOYPMVU9Ms1M1MkgeE+WgU5s2'},
  'text': "I do not know your current local time because I don't have access "
          "to your device's clock or location. \n"
          '\n'
          'However, you can easily check the time in the top right or bottom '
          'right corner of your screen!',
  'type': 'text'}]


The model has no clock. Its weights were frozen months ago and nothing in the context tells it the time. It can only decline or guess.

We can surpass this limitations by augmenting the LLM with a tool.

## 2. Defining a tool

A tool starts life as an ordinary Python function.

In [5]:
from datetime import datetime

def get_current_time():
    """Return the current local date and time."""
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

In [6]:
get_current_time()

'Friday, September 18, 2026 at 04:43 PM'

### Requirements for Tool Definition
To make this usable by a model, LangChain needs three things: 
- a name (start with a verb and describe the core of the task)
- a description in the docstring
- an argument schema. 

The `@tool` decorator builds all three from your function. 

**The docstring is not a comment here; the model reads it to decide when to call the tool. Write it carefully.**

In [7]:
# import the tool decorator from langchain's tools module
from langchain.tools import tool

#TODO: define `get_current_time` as a tool
@tool
def get_current_time() -> str:
    '''
    Return the current local date and time.
    '''
    return datetime.now().strftime("%A, %B %d, %Y at %I:%M %p")

Inspect what the decorator built:

In [8]:
print(get_current_time.name)
print(get_current_time.description)
print(get_current_time.args)

get_current_time
Return the current local date and time.
{}


In [9]:
get_current_time.invoke({})

'Friday, September 18, 2026 at 04:43 PM'

## 3. Give the tool to the model

`bind_tools` attaches tool schemas to the model as a **list** of tools.

The model does not gain the ability to run anything. It only gains the ability to ask.

In [10]:
#TODO: bind the tool to the model
model_with_tools = model.bind_tools([get_current_time])

In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What time is it right now?")
response = model_with_tools.invoke([question])

In [12]:
pprint(response)

AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_95873': 'El4KXAFpFH0Tg4cSl/6JRPLL2znNKewW/FMKkA3JWKak5j52j4Cj8mH8loxVSUps2WFmfbyGsHg/NPEOagmHovE4zIwsJtoQDGZa2iCOkaSKR7UQEsQtluLl+Plm2u9f'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b642-6d44-7291-9c69-08213295625d-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_95873', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 31, 'output_tokens': 12, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}})


In [13]:
response.pretty_print()

================================== Ai Message ==================================

[]
Tool Calls:
  get_current_time (call_95873)
 Call ID: call_95873
  Args:


Note: `content` is empty. Instead of answering, the model produced a `tool_calls` entry naming our tool.

Remember the model proposes; it never executes.

In [14]:
pprint(response.response_metadata)

{'finish_reason': 'STOP',
 'model_name': 'gemini-3.5-flash-lite',
 'model_provider': 'google_genai',
 'safety_ratings': []}


In [15]:
print(response.tool_calls)

[{'name': 'get_current_time', 'args': {}, 'id': 'call_95873', 'type': 'tool_call'}]



Executing is our job. Run the tool with the call the model requested:

In [16]:
call = response.tool_calls[0]
tool_result = get_current_time.invoke(call)
print(tool_result)

content='Friday, September 18, 2026 at 04:43 PM' name='get_current_time' tool_call_id='call_95873'


That produced a `ToolMessage`. Now we can combine the full conversation, including the model's tool call and the tool's result, and send it back:

In [17]:
messages = [question, response, tool_result]

final = model_with_tools.invoke(messages)


In [18]:
pprint(final)

AIMessage(content=[{'type': 'text', 'text': 'It is currently 4:43 PM on Friday, September 18, 2026.', 'extras': {'signature': 'El4KXAFpFH0Tqi/nQl7ekWsK44zarEmfOjwB27Fl3iqUOJAaEh3M2Fuh5M5+9noA0jBaFAqEwNiz1spF9GxRIVoRqzDHRlkc2Pr4MMttF3uraGr21NVd/z9vJ0uJgY/s'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b642-8036-7a13-bb16-3cb9c82fa2f0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 23, 'total_tokens': 101, 'input_token_details': {'cache_read': 0}})


In [19]:
print(final.content)

[{'type': 'text', 'text': 'It is currently 4:43 PM on Friday, September 18, 2026.', 'extras': {'signature': 'El4KXAFpFH0Tqi/nQl7ekWsK44zarEmfOjwB27Fl3iqUOJAaEh3M2Fuh5M5+9noA0jBaFAqEwNiz1spF9GxRIVoRqzDHRlkc2Pr4MMttF3uraGr21NVd/z9vJ0uJgY/s'}}]


### Interim Summary

1. The model saw the question and proposed a tool call.
2. Our code executed the tool.
3. The result went back into the conversation.
4. The model read the result and answered.

Model proposes, runtime executes, result returns. That loop is an agent.

## 4. `create_agent`: Defining the Agentic Loop

The loop so far looked like this:
 propose, execute, return, repeat until the model answers. 
 
 Because this is such a common pattern, LangChain provides a dedicated method to run the loop: `create_agent`
 
 - https://reference.langchain.com/python/langchain/agents/factory/create_agent

 Basic signature: `create_model(model, tools, system_prompt)`


In [20]:
system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""

In [21]:
from langchain.agents import create_agent

#TODO: define an agent with model, tool list, and system prompt
agent = create_agent(
    model=model,
    tools=[get_current_time],
    system_prompt= system_prompt,
)



In [22]:
result = agent.invoke({"messages": question})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

What time is it right now?
================================== Ai Message ==================================

[]
Tool Calls:
  get_current_time (call_133367)
 Call ID: call_133367
  Args:
================================= Tool Message =================================
Name: get_current_time

Friday, September 18, 2026 at 04:43 PM
================================== Ai Message ==================================

[{'type': 'text', 'text': 'It is currently 4:43 PM on Friday, September 18, 2026.', 'extras': {'signature': 'El4KXAFpFH0Ti5VD3JaW2HrjRDJPqBtI5pR4ahEX7WBLHWy4huqK4NzV1ypvgy7gnPUB/6q23HIilZafxwm8fjl+1gHiopPvCzwgJn0tfTIWxtPXwXoPpSZVbAEdu3v7'}}]


## 6. Multiple tools

Real agents have several tools and choose between them. Models are bad at date arithmetic, so we'll give our agent a second tool for that.

In [23]:
@tool
def count_days_until_given_date(date: str) -> str:
    """Return the number of days from today until a future date given as YYYY-MM-DD."""
    target = datetime.strptime(date, "%Y-%m-%d").date()
    delta = (target - datetime.now().date()).days
    return f"{delta} days"


In [24]:
agent = create_agent(model=model, 
                    tools=[get_current_time,count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )


In [25]:
question_day = HumanMessage(content="How many days until the midterm on 2026-10-30?")
response = agent.invoke({"messages":question_day })


In [26]:
pprint(response)

{'messages': [HumanMessage(content='How many days until the midterm on 2026-10-30?', additional_kwargs={}, response_metadata={}, id='5d2dc95b-e675-4120-ba2a-909f033ff1b2'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'count_days_until_given_date', 'arguments': '{"date": "2026-10-30"}'}, '__gemini_function_call_thought_signatures__': {'call_173398': 'El4KXAFpFH0Ta5407MQv/96McnIyQt/d5e8V9KoktWmMEWCxq0Gr7AKKRu3HIKxtM2aYkKXGV3F+iovTmLgQSrkoR0JujWBvgwupfFmfsfCpPrtfRadEwHMDd+BcBMJD'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b642-9adf-7ab0-9f4f-9425762f1875-0', tool_calls=[{'name': 'count_days_until_given_date', 'args': {'date': '2026-10-30'}, 'id': 'call_173398', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 31, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}}),
           

The model picked the right tool and filled the argument from the question. Since executing calls is always the same steps, write a small helper:

## 6. Tools that call real APIs

**Motivation for web search:**
GPT-4.1-mini has a knowledge cutoff date of June 1, 2024.
 - https://developers.openai.com/api/docs/models/gpt-4.1-mini

 Qwen3.5's knowledge cutoff date is late 2024 to early 2025.

 Gemini 3.1 Flash-Lite has a knowledge cutoff date of January 2025.

They won't be able to answer questions about events that may have happened after the cutoff date

In [27]:
# same agent as before
agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date],
                    system_prompt = """You're a helpful assistant who answers users' questions concisely. 
                    Use the tools available when necessary."""
                    )

In [28]:
questions = [HumanMessage("Which country won the FIFA World Cup 2026"), 
            HumanMessage("Who won the Super Bowl 2026?"), 
            HumanMessage("Who is the current president of the US")]

In [29]:
result = agent.invoke({"messages": questions[-1]})
pprint(result)


{'messages': [HumanMessage(content='Who is the current president of the US', additional_kwargs={}, response_metadata={}, id='9b51f76d-67ae-45c5-955b-121a6b4e6697'),
              AIMessage(content=[{'type': 'text', 'text': 'As of my current knowledge base, Joe Biden is the President of the United States. Please note that my data is current up to January 2025, and I do not have access to real-time information regarding events or changes that occurred after that date.', 'extras': {'signature': 'El4KXAFpFH0TqDzm8FdDtXETvbAtuk3wto99FmhaEpHdlS84ekeoSkHY5BuZDEeRILOus/iKGkA9eaPwR25t65UpvLH0R25q3qOkj/let0h5vSlqyiPfU7TUAJqwlia9'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b642-af94-7d11-8aac-f690f245ac20-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 54, 'total_tokens': 173, 'input_token_details': {'cache_read'

In [30]:
pprint(result["messages"][-1].content)

[{'extras': {'signature': 'El4KXAFpFH0TqDzm8FdDtXETvbAtuk3wto99FmhaEpHdlS84ekeoSkHY5BuZDEeRILOus/iKGkA9eaPwR25t65UpvLH0R25q3qOkj/let0h5vSlqyiPfU7TUAJqwlia9'},
  'text': 'As of my current knowledge base, Joe Biden is the President of the '
          'United States. Please note that my data is current up to January '
          '2025, and I do not have access to real-time information regarding '
          'events or changes that occurred after that date.',
  'type': 'text'}]


In [31]:
# loop through questions and then invoke the agent
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    print("")

'Which country won the FIFA World Cup 2026'
[{'extras': {'signature': 'El4KXAFpFH0TjeNjPP7F4I82yUq5A0z4OCJ5bPxpwKnh+nkjQxMqPggNJEnnJKwE+0p3kLqWxrTtnZayhcJdH13ZvLITLCgVQbmPyfC6DjgpFipra5DG6OU/1XKgUnCH'},
  'text': 'The 2026 FIFA World Cup has not taken place yet; it is scheduled to '
          'be held from June 11 to July 19, 2026.',
  'type': 'text'}]

'Who won the Super Bowl 2026?'
[{'extras': {'signature': 'El4KXAFpFH0TRb9KVRvq0XnEKXduNLAbgXEO5BcEqBrqCqoGKeTtPZI/bwn7j4Y2WIwBhS9H92f89rr22+L9CziLq0vNJPBM1N2OIUGAomSkgBxQK/c55xx4RKKzCVQU'},
  'text': 'Super Bowl LX (60) was played on February 8, 2026, and was won by '
          'the **Seattle Seahawks**, who defeated the **New York Jets** with a '
          'score of 29–13.',
  'type': 'text'}]

'Who is the current president of the US'
[{'extras': {'signature': 'El4KXAFpFH0TnYUB+FpBB47oHWrOlfebK6zXoQhuglssPsCS+kfnxuGFaSDlsKu2csFgEPCflQ4ngeQ7CRFF32R7YM3usGfl+IS6CglaKkjyr4nP+9luseypwie/Ge4a'},
  'text': 'As of my current knowledge base, J

### Tavily Web Search API

Let's add web search capability to our agent. We'll use Tavily as our web search API.
- https://app.tavily.com/

We'll need to provide an API key in the `.env` file, which has a `TAVILY_API_KEY` field.

#### TavilySearch
Because Tavily is specialized for agentic web search, LangChain supports it.

The `TavilySearch` method queries the Tavily Search API and gets back json.

https://reference.langchain.com/python/langchain-tavily/tavily_search/TavilySearch


In [32]:
from langchain_tavily import TavilySearch
from typing import Dict, Any

@tool
def search_the_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    tavily = TavilySearch(max_results=3)
    query_dict = {"query": query}
    results = tavily.invoke(query_dict)
    return results


In [33]:
data = search_the_web.invoke({"query":"Who won the super bowl 2026"})

In [34]:
search_docs = data.get("results", data)


In [35]:
pprint(search_docs)

[{'content': 'After a long, lonely offseason, you may have forgotten where the '
             'NFL drama left off after the last big game. While all eyes are '
             'focused on making it to Super Bowl 61 – which is still months '
             "away – here's a little refresher to get back up to speed.\n"
             '\n'
             '## Who won Super Bowl 2026?\n'
             '\n'
             'The Seattle Seahawks defeated the New England Patriots, 29-23, '
             "in Super Bowl 60, which took place at Levi's Stadium in Santa "
             'Clara, California, on Feb. 8, 2026. [...] Seattle Seahawks '
             'defeated the New England Patriots, 29-23, in Super Bowl 60.\n'
             '\n'
             '## Who won Super Bowl 60 MVP?\n'
             '\n'
             'Seattle Seahawks running back Kenneth Walker III was named the '
             'MVP of Super Bowl 60.\n'
             '\n'
             '## Who won Super Bowl 2025?\n'
             '\n'
             'T

Our agent has three tools now:

In [37]:
from langchain.agents import create_agent

agent = create_agent(model=model, 
                    tools=[get_current_time, count_days_until_given_date, search_the_web],
                    system_prompt = "You're a helpful assistant who answers users' questions concisely. Use the tools available when necessary."
)

In [38]:
result = agent.invoke({"messages": questions[0]})

In [39]:
pprint(result)

{'messages': [HumanMessage(content='Which country won the FIFA World Cup 2026', additional_kwargs={}, response_metadata={}, id='e7763f22-d288-4a18-b5c3-5da9d73d928c'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_170380': 'El4KXAFpFH0TDEdC8WEd/dMKYG9C2b3dBkaQxK5bs8JeaGAB+w2EcwWiGbDXJ9lvjEukZ5EOfNjOIGOshSXb1k94N3Xys0hb57P+ivmG9I6kPFUjG5kfgWtmgDRvbdhf'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0b643-abd2-72f0-9914-d5f9b47f3d77-0', tool_calls=[{'name': 'get_current_time', 'args': {}, 'id': 'call_170380', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 164, 'output_tokens': 12, 'total_tokens': 176, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='Friday, September 18, 2026 at 04:44 PM', na

In [40]:
for q in questions:
    result = agent.invoke({"messages": q})
    pprint(result["messages"][0].content)
    pprint(result["messages"][-1].content)
    

'Which country won the FIFA World Cup 2026'
[{'extras': {'signature': 'El4KXAFpFH0TrWIA6odnvAK0GphqRe62x30pbiglr4VahOxske85tkHV/WIsynERrC20ybyvVm2SpH3+GeoR0AFk6jA416OdbkQSg13mpmp7eNMMIW84vJH/NGBl/aT8'},
  'text': 'Spain won the 2026 FIFA World Cup, defeating Argentina in the '
          'final.',
  'type': 'text'}]
'Who won the Super Bowl 2026?'
[{'extras': {'signature': 'El4KXAFpFH0TrXLdbWouISl4bYYdJsV0egXWil5il/yROAX3U5j6l7KAAw6Iq5wr1iNGjZ9SI/6klfWSKtzxkuCZicCrPzovSMfGDcSSr1/3Mgcg32tFOXlqtSYdcrl7'},
  'text': 'The **Seattle Seahawks** won Super Bowl LX (2026), defeating the '
          'New England Patriots 29–13.',
  'type': 'text'}]
'Who is the current president of the US'
[{'extras': {'signature': 'El4KXAFpFH0TOfQp80OC55D/e4gYEtSkBEvh/PuS7//++TleE4BrwWq1y4Z3Aa0++jFtGM542BsUMn2Tyre/ZnAQqa20iK/rCfawfrA0KS8Hom9WyymuPoOa/d/UT7g3'},
  'text': 'The current president of the United States is Donald Trump. He is '
          'the 47th president, having taken office on January 20, 2025.',
  

In [41]:
# the response from the model
pprint(result["messages"][-1].content)

[{'extras': {'signature': 'El4KXAFpFH0TOfQp80OC55D/e4gYEtSkBEvh/PuS7//++TleE4BrwWq1y4Z3Aa0++jFtGM542BsUMn2Tyre/ZnAQqa20iK/rCfawfrA0KS8Hom9WyymuPoOa/d/UT7g3'},
  'text': 'The current president of the United States is Donald Trump. He is '
          'the 47th president, having taken office on January 20, 2025.',
  'type': 'text'}]


## 7. ICA: Build a weather tool

Your turn. [wttr.in](https://wttr.in) is a weather service that takes a city name directly and needs no key. One request:

```
https://wttr.in/Boston?format=j1
```

returns JSON. 

The current conditions live in `data["current_condition"][0]`, with fields including `temp_F`, `weatherDesc`, and `windspeedMiles`.

Boston or Boston,MA are acceptable. If city name matches multiple locations, it'll likely default to the one closest to your IP address.

### Your task:
Write a tool `get_weather(city)` that returns the current conditions for that city. Steps:

1. Build the URL from the city argument and fetch it with `requests.get(...).json()`.
2. Pull out temperature, conditions, and wind, and return a readable string.
3. Write the docstring so the model knows when to reach for this tool.
4. Add it to the agent and ask a question that needs it.

Skeleton:

In [42]:
import requests
data = requests.get("https://wttr.in/Boston?format=j1").json()

In [43]:
pprint(data["current_condition"][0])

{'FeelsLikeC': '23',
 'FeelsLikeF': '73',
 'cloudcover': '100',
 'humidity': '46',
 'observation_time': '08:38 PM',
 'precipInches': '0.0',
 'precipMM': '0.0',
 'pressure': '1015',
 'pressureInches': '30',
 'temp_C': '24',
 'temp_F': '76',
 'uvIndex': '2',
 'visibility': '10',
 'visibilityMiles': '6',
 'weatherCode': '122',
 'weatherDesc': [{'value': 'Overcast '}],
 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}],
 'winddir16Point': 'NNW',
 'winddirDegree': '346',
 'windspeedKmph': '11',
 'windspeedMiles': '7'}


In [49]:
import requests

@tool
def get_weather(city: str) -> str:
    """TODO: describe what this tool does and what the argument means."""
    # TODO: fetch https://wttr.in/<city>?format=j1 with requests.get
    # TODO: parse the JSON and pull out current_condition and return this segment of the dictionary
    
    data = requests.get(f"https://wttr.in/{city}?format=j1").json()
    return data["current_condition"][0]

In [52]:
# Test your tool directly first:
print(get_weather.invoke({"city": "Boston"}))

# Then give it to the agent:
agent = create_agent(model=model, tools=[get_current_time, count_days_until_given_date, search, get_weather])
result = agent.invoke({"messages": [HumanMessage(content="Should I bring a jacket in Boston tonight?")]})
print(result["messages"][-1].content)

{'FeelsLikeC': '23', 'FeelsLikeF': '73', 'cloudcover': '100', 'humidity': '46', 'observation_time': '08:38 PM', 'precipInches': '0.0', 'precipMM': '0.0', 'pressure': '1015', 'pressureInches': '30', 'temp_C': '24', 'temp_F': '76', 'uvIndex': '2', 'visibility': '10', 'visibilityMiles': '6', 'weatherCode': '122', 'weatherDesc': [{'value': 'Overcast '}], 'weatherIconUrl': [{'value': 'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png'}], 'winddir16Point': 'NNW', 'winddirDegree': '346', 'windspeedKmph': '11', 'windspeedMiles': '7'}


NameError: name 'search' is not defined